# MOSS joint transcription and diarization

Run top to bottom in a **fresh local Jupyter kernel** on Alex's Linux 5090 machine. Install the matching environment from README first. Set `INPUT_FILE` to one local audio/video file. `DENOISE=True` applies FFmpeg `afftdn`; `False` copies the normalized WAV unchanged. Both versions and the input hash are saved under `runs/`.

Outputs are draft anonymous speaker labels, not verified identities or clinical roles. Check the audio before interpreting them. No input recordings or outputs belong in Git.


## 1. Settings and GPU check

In [ ]:
ROUTE = "moss"
from pathlib import Path
import sys
from pipeline_common import prepare_audio, gpu_preflight, save_turns, save_summary

INPUT_FILE = "/absolute/path/to/your/audio_or_video.mp4"  # EDIT
OUTPUT_ROOT = "runs"
AUDIO_STREAM = 0
DENOISE = False  # True / False
NOISE_REDUCTION_DB = 12  # FFmpeg afftdn nr; 1..30; only used if DENOISE=True

if INPUT_FILE.startswith("/absolute/path/to/"):
    raise ValueError("Set INPUT_FILE to an existing local audio/video file before Run All")
print("Notebook Python:", sys.executable)
print("Route:", ROUTE, "| denoise:", DENOISE)


## 2. Convert video/audio to WAV; optionally denoise

In [ ]:
run_dir, model_wav, manifest = prepare_audio(
    INPUT_FILE, OUTPUT_ROOT, ROUTE, denoise=DENOISE,
    audio_stream=AUDIO_STREAM, noise_reduction_db=NOISE_REDUCTION_DB,
)
duration_seconds = manifest["duration_seconds"]
print(f"Model input: {model_wav} ({duration_seconds:.2f}s)")
print(f"Denoising: {manifest['denoise_filter'] or 'off'}")
print(f"Output: {run_dir}")


## 3. Infer and save results

In [ ]:
import json
import time
device, gpu_info = gpu_preflight()
print(gpu_info)
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from moss_transcribe_diarize import parse_transcript
from moss_transcribe_diarize.inference_utils import build_transcription_messages, generate_transcription

MODEL_ID = "OpenMOSS-Team/MOSS-Transcribe-Diarize"
MAX_NEW_TOKENS = 8192
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype="auto", attn_implementation="sdpa",
).to(dtype=dtype).to(device).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
started = time.perf_counter()
result = generate_transcription(
    model, processor, build_transcription_messages(model_wav),
    max_new_tokens=MAX_NEW_TOKENS, do_sample=False, device=device, dtype=dtype,
)
elapsed = time.perf_counter() - started
raw_text = result["text"]
(run_dir / "moss_raw.txt").write_text(raw_text, encoding="utf-8")
(run_dir / "moss_raw.json").write_text(json.dumps(result, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
segments = [dict(start_seconds=s.start, end_seconds=s.end,
                 model_speaker=s.speaker, text=s.text)
            for s in parse_transcript(raw_text)]
rows = save_turns(run_dir, segments, duration_seconds)
save_summary(run_dir, model=MODEL_ID, **gpu_info, denoise=DENOISE,
             generated_tokens=result.get("generated_tokens"), max_new_tokens=MAX_NEW_TOKENS,
             inference_seconds=round(elapsed, 2), parsed_turns=len(rows),
             needs_review=not rows or result.get("generated_tokens", 0) >= MAX_NEW_TOKENS)
print(f"{len(rows)} turns saved in {run_dir}; inspect moss_raw.txt if empty or truncated")
rows[:10]
